# Normalizacja tekstu

Dzisiaj zajmiemy się pracą z "surowym" tekstem zawierającym szereg zanieczyszczeń których będziemy się chcieli pozbyć.

W pierwszym kroku zapoznamy się jednak z zasobem jakim jest Słownik Gramatyczny Języka Polskiego (https://sgjp.pl/leksemy/), a w szczególności jego udostępniana publicznie reprezentacja w postaci kolumnowej, dostępna w ramach narzędzia Morfeusz (https://morfeusz.sgjp.pl/).

Reprezentacja kolumnowa oznacza podobną organizację danych do tej którą znamy już z formatów **conllu** lub **csv**. W tym wypadku separatorem jest znak tabulacji: `\t`. Możemy więc skorzystać z biblioteki `pandas` do wczytania słownika jako tabeli. Wiersze oznaczają kolejno: 
1. formę tekstową
2. lemat - formę słownikową
3. znacznik morfosyntaktyczny
4. oznaczenia nazw własnych
5. dodatkowe kwalifikatory

In [ ]:
import pandas
from pathlib import Path

DATA_DIR = Path("../../data")

dict_path = DATA_DIR / "others" / "sgjp-20260823.tab"
dictionary = pandas.read_csv(dict_path, sep="\t", skiprows=list(range(28)), header=None, names=("form", "lemma", "tag", "ner", "qual"))
dictionary

Pierwszym krokiem który wykonamy, będzie redukcja słownika do zbioru unikalnych form tekstowych:

In [ ]:
vocab = set(dictionary["form"])
print(f"{len(vocab)} unique forms")

Spójrzmy więc na materiał, z którym będziemy dzisiaj pracowali: jest to tekst powieści Lalka Bolesława Prusa, uzyskany poprzez procedurę *Optical Character Recognition* (OCR) na podstawie PDFa udostępnionego przez serwis https://wolnelektury.pl

In [ ]:
def read_file(fpath): # helper function
    with open(fpath) as f:
        txt = f.read()
    return txt

raw_lalka = read_file(DATA_DIR / "lektury" / "lalka.txt")
print(raw_lalka[:7000])

Jak widzimy, jakość tekstu pozostawia wiele do życzenia. Poznamy dzisiaj kilka technik, które są pomocne przy przygotowaniu - preprocessingu tekstów na poczet późniejszych prac. Najprawdopodobniej nie uda nam się usunąć wszystkich zanieczyszczeń, ale dobrym podejściem jest sukcesywne eliminowanie tych, które najbardziej rzucają się w oczy przez ich częstość. Podzielmy więc tekst prostym tokenizatorem, i wygenerujmy listę frekwenycjną form występujących w tekście:

In [ ]:
import re

tokenizer = re.compile(r"\b\S+\b")
tokens = tokenizer.findall(raw_lalka)
df = pandas.DataFrame({"token": tokens})
df["token"].value_counts().to_frame().iloc[0:30]

Pierwsza obserwacja:

Do tekstu powieści dostały się elementy metatekstowe - przypisy redakcyjne:

In [ ]:
footnote_index = raw_lalka.index("redakcy�ny")
margin = 400
print(raw_lalka[footnote_index-margin:footnote_index+margin])

Widzimy, że zachowują się one dość regularnie, zdefiniujmy wyrażenie regex pozwalający na wykrycie ich w tekście:

In [ ]:
footnote_re = re.compile(r"^.+redakcy�ny\] $", flags=re.MULTILINE)
footnote_re.findall(raw_lalka)

Możemy usunąć te wszystkie wystąpienia metodą `sub` obiektu `re.Pattern`:

In [ ]:
clean_lalka = footnote_re.sub("", raw_lalka)
print(f"Reduction: {len(clean_lalka)}/{len(raw_lalka)}", "\n")
print(clean_lalka[footnote_index-margin:footnote_index+margin])

Druga obserwacja:

Teksty wydawane przez Wolne Lektury są często anotowane motywami reprezentowanymi w tekście, np. tutaj mamy

`Alkohol, Pĳaństwo`

Najczęśćiej są one zapisane jako przedzielone przecinkami frazy występujące w osobnej linii. Zobaczmy listę najczęściej występujących w powieści linii:

In [ ]:
lines = clean_lalka.split("\n")
line_df = pandas.DataFrame({"line": lines})
line_counts = line_df.value_counts()
line_counts.to_frame().iloc[:50]

Widzimy kilka godnych odnotowania wyjątków, poza którymi niemal wszystkie z wypisanych linii stanowią element obcy w powieści. Usuńmy je:

In [ ]:
exceptions = [
    "",
    "— Tak. ",
    "Wokulski zamyślił się. ",
    "Wokulski uśmiechnął się. "  
]

for motif_phrase in line_counts.index[:50]:
    motif_phrase = motif_phrase[0]
    if motif_phrase not in exceptions:
        motif_re = re.compile(r"^" + re.escape(motif_phrase) + r"$", flags=re.MULTILINE)
        clean_lalka = motif_re.sub("", clean_lalka)

print(f"Reduction: {len(clean_lalka)}/{len(raw_lalka)}", "\n")

## Ćwiczenie 1
Jak widzieliśmy linie sygnalizujące motywy są często przedzieloną przecinkami konkatenacją indywidualnych motywów. Zaproponuj wyrażenie regularne które wychwyci takie linie na podstawie wyliczenia indywidualnych motywów i sposobu ich połączenia w linię, jakiej przykłady obserwujemy w tekście

In [ ]:
# TODO

Wykorzystajmy teraz słownik, do identyfikacji tokenów, które do niego nie pasują - tokeny takie nazywamy **Out of Vocabulary** (OOV):

In [ ]:
clean_tokens = tokenizer.findall(clean_lalka)
clean_lalka_df = pandas.DataFrame({"token": clean_tokens})
clean_lalka_df["oov"] = ~(clean_lalka_df["token"].isin(vocab))
print(clean_lalka_df["oov"].value_counts(normalize=True))
clean_lalka_df

Ponownie, interesują nas najczęściej występujące tokeny tego typu:

In [ ]:
oov = clean_lalka_df[clean_lalka_df["oov"]]
oov.value_counts().iloc[:50]

Znak "j" został niepoprawnie rozpoznany na etapie OCR, i jest zapisywany jako "�". Poprawmy to, i powtórzmy analizę:

In [ ]:
clean_lalka = clean_lalka.replace("�", "j")
clean_tokens = tokenizer.findall(clean_lalka)
clean_lalka_df = pandas.DataFrame({"token": clean_tokens})
clean_lalka_df["oov"] = ~(clean_lalka_df["token"].isin(vocab))
print(clean_lalka_df["oov"].value_counts(normalize=True))
oov = clean_lalka_df[clean_lalka_df["oov"]]
oov.value_counts().iloc[:50]

## Ćwiczenie 2:

Na trzeciej pozycji listy słów spoza słownika dalej znajduje się słowo "barǳo", spróbuj zidentyfikować przyczynę, i wyeliminować ją

In [ ]:
# TODO

## Ćwiczenie 3:

Słownik zawiera klucze o określonej wielkości znaków - poza nazwami własnymi i akronimami są to w większości formy pisane minuskułą. Jak wpływa to na status OOV? Sprawdź jakie będą najczęstsze tokeny OOV po wzięciu pod uwagę tej komplikacji

In [ ]:
# TODO

# Lematyzacja

Często możemy chcieć uprościć część złożoności w zakresie form z jakimi mamy do czynienia. Procedura ta może przybierać różne formy.

Język angielski charakteryzuje się ubogą morfologią, w jego wypadku formy odmiany czasowników (przez czas gramatyczny), rzeczowników (przez liczbę) i przymiotników (przez stopień) są bardzo regularne, i polegają w większości wypadków na sufiksacji - dołączaniu odpowiednich końcowek do wyrazu. W takim przypadku efektywny będzie *stemming* czyli obcinanie form tekstowych do ich tematu fleksyjnego.

Język polski wygląda diametralnie odmiennie: paradygmaty odmiany rzeczowników, czasowników i przymiotników są bardzo złożone, i liczne. Często trudno jest wyodrębnić temat, który pozwalałby jednoznacznie zidentyfikować słowo. Dlatego też w języku polskim najczęściej stosuje się procedurę *lematyzacji* czyli sprowadzenia słów występujących w tekście do ich postaci słownikowej. Lematyzacja ma za zadanie redukcję złożoności *fleksyjnej* - i.e. związanej z gramatyką, a nie słowotwórczej Słownik SGJP pozwala na przygotowanie narzędzia mapującego formy tekstowe na lematy:

In [ ]:
from tqdm import tqdm

lemmatizer_dict = {}
pairs = dictionary[["form", "lemma"]].itertuples()
for tup in tqdm(pairs):
    form, lemma = tup.form, tup.lemma
    if form in lemmatizer_dict:
        lemmatizer_dict[form].append(lemma)
    else:
        lemmatizer_dict[form] = [lemma]

# equivalent, more elegant, but much slower:
#lemmatizer = dictionary.groupby("form").agg({"lemma": list})["lemma"]
#lemmatizer_dict = lemmatizer.to_dict()

In [ ]:
for form in ["dzieciątkami", "widzielibyśmy", "najpiękniejszemu"]:
    print(f"{form} -> {lemmatizer_dict[form]}")

Zwróćmy uwagę, że mapowanie to przypisuje formom *listy* lematów, i.e. wartości takich może być więcej niż jedna. Jest tak dlatego, że formy tekstowe są często niejednoznaczne, i dopiero na podstawie kontekstu wiemy które ze słów słownikowych jest reprezentowane przez dany ciąg znaków. W ekstremalnych przypadkach, jedna forma może korespondować z lematami należącymi do innych części mowy:

In [ ]:
for form in ["ci", "tony", "mai", "stoi"]:
    print(f"{form} -> {lemmatizer_dict[form]}")

Procedurę lematyzacji słownikowej możemy uzupełnić etapem dezambiguacji, która powoli wybrać najbardziej prawdopodobną interpretację. Prosty algorytm realizujący takie zadanie wykorzystuje tagi morfosyntaktyczne (zakładając że tekst jest otagowany). Spośród lematów zgadzających się co do morfologii (m.in. części mowy) z naszym słowem, np.:

"Pokażę *ci* jak brzmią te dwa *tony*"

1. "ci" jest dopełnieniem bliższym "Pokażę", więc musi być rzeczownikiem w celowniku, to wskazuje na lemat "ty".
2. "dwa" sygnalizuje jeden z rodzajów męskich, zatem "tony" musi być postacią "ton"

Jeżeli po filtrowaniu morfosyntaktycznym dalej mamy więcej niż jeden wariant do wyboru, możemy, posługując się danymi frekwencyjnymi wybrać najczęściej reprezentowany lemat. W naszym podejściu nie mamy dostępu do żadnego z tych zasobów, więc zadowolimy się klasami lematów:

In [ ]:
clean_lalka_df["lower_token"] = clean_lalka_df["token"].apply(lambda x: x.lower())
clean_lalka_df["lemmas"] = clean_lalka_df["lower_token"].map(lemmatizer_dict)
clean_lalka_df["num_lemmas"] = clean_lalka_df["lemmas"].apply(lambda x: len(x) if x == x else 0) # handling NAN values
clean_lalka_df

Zobaczmy najbardziej niejednoznaczne słowa:

In [ ]:
clean_lalka_df.sort_values("num_lemmas", ascending=False).drop_duplicates("token").iloc[:20]

## Ćwiczenie 4:

Zastanów się jakie jeszcze zanieczyszczenia widzisz w tekście Lalki, i jak - korzystając z poznanych narzędzi - możesz je wyeliminować. Sprawdź jak wpłynie to na długość oczyszczonego tekstu, oraz proporcję słów spoza słownika